In [1]:
from pathlib import Path
import glob
import os
from tqdm import tqdm
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
import re
from openslide import OpenSlide
from functools import cached_property
import geopandas as gpd
import pandas as pd

DATA_DIR = Path("/data") / "irb" / "pharmacometrics" / "pro00117942"
DUKE_HOVERNEXT_PREDICTIONS_DIR = DATA_DIR / "hover_next_predictions"
DUKE_SLIDES_DIR = DATA_DIR / "digitized_slides"
INOVA_HOVERNEXT_PREDICTIONS_DIR = DATA_DIR / "hover_next_predictions_inova"
INOVA_SLIDES_DIR = DATA_DIR / "digitized_slides_inova"

class PreAnnotatedSlide:
    def __init__(self, path_to_svs : Path, model_predictions_root : Path):
        if not path_to_svs.exists() or not model_predictions_root.exists(): raise ValueError("Please ensure both paths are valid and exist")
        self.model_predictions_root = model_predictions_root
        self.path_to_svs = path_to_svs

    @cached_property
    def slide(self):
        return OpenSlide(self.path_to_svs)

    @cached_property
    def cells(self):
        geojson_filepath = self.model_predictions_root / "poly.geojson"
        # geopandas is designed for geographic data, CRS of None for microscope data
        cells = gpd.read_file(geojson_filepath).set_crs(crs=None, allow_override=True)
        cells['centroid_x'] = cells.centroid.x
        cells['centroid_y'] = cells.centroid.y
        return cells    

def extract_study_id(slide_file_stem):
    pattern = re.compile(r'DUK-(DU-R-\d{4})-(R|RC|le|LC|C|Le)')
    match = re.fullmatch(pattern, slide_file_stem)
    assert match is not None, f"{slide_file_stem} did not match {pattern}"
    return match.group(1) 

## Feature Calculation

In [18]:
from sklearn.cluster import KMeans, MiniBatchKMeans, OPTICS, DBSCAN, HDBSCAN

NOISY_DATAPOINT_CLUSTER_LABEL = -1
MIN_INFLAMMATORY_CELLS = 5

class TooFewCellsException(Exception):
    pass

def cluster_inflammatory_cells(cells):
    inflammatory_cell_centroids = cells[['centroid_x', 'centroid_y']].copy()
    if len(inflammatory_cell_centroids) < MIN_INFLAMMATORY_CELLS: raise TooFewCellsException()
    model = HDBSCAN(copy=True, min_cluster_size=MIN_INFLAMMATORY_CELLS)
    cluster_labels = model.fit_predict(inflammatory_cell_centroids.to_numpy())
    inflammatory_cell_centroids['cluster_label'] = cluster_labels
    inflammatory_cell_clusters = pd.merge(cells, inflammatory_cell_centroids, on=["centroid_x", "centroid_y"])
    return inflammatory_cell_clusters

def estimate_cluster_areas(clusters, microns_per_pixel):
    cluster_estimated_areas = []
    
    for cluster_label in clusters['cluster_label'].unique():
        if cluster_label == NOISY_DATAPOINT_CLUSTER_LABEL: continue
        cluster_members = clusters[clusters['cluster_label'] == cluster_label]
        cluster_member_vertices = cluster_members.geometry.extract_unique_points()
        hull = cluster_member_vertices.convex_hull
        area_microns_sq = (hull.area * (microns_per_pixel ** 2)).to_numpy().tolist()
        cluster_estimated_areas.extend(area_microns_sq)
    return cluster_estimated_areas
    

In [19]:
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu

def calculate_tissue_area(slide : OpenSlide):
    level = slide.level_count - 1
    dims = slide.level_dimensions[level]
    downsample = slide.level_downsamples[level]
    region = slide.read_region((0, 0), level, dims)
    img_rgb = np.array(region.convert("RGB"))
    gray = rgb2gray(img_rgb)
    thresh = threshold_otsu(gray)
    tissue_mask = gray < thresh
    tissue_pixels_at_level = np.sum(tissue_mask)
    pixel_area_level0 = tissue_pixels_at_level * (downsample**2)

    microns_per_pixel = float(slide.properties['aperio.MPP'])
    return pixel_area_level0 * (microns_per_pixel ** 2)

In [20]:
from sklearn.neighbors import KDTree
from scipy.stats import gmean

EPSILON = np.finfo(np.float64).eps

def calculate_avg_inflammatory_cell_cluster_size(slides : list[PreAnnotatedSlide]) -> float:
    inflammatory_cell_cluster_areas = []
    for slide in slides:
        try:
            clusters = cluster_inflammatory_cells(slide.cells)
            estimated_areas = estimate_cluster_areas(clusters, float(slide.slide.properties['aperio.MPP']))
            inflammatory_cell_cluster_areas.extend(estimated_areas)
        except TooFewCellsException:
            continue

    # this distribution is roughly gaussian in the log space, 
    # can use g-mean or median here for a good measure of central tendency 
    return gmean(inflammatory_cell_cluster_areas) if inflammatory_cell_cluster_areas else 0.0

def calculate_relative_abundance(slides : list[PreAnnotatedSlide], cell_type : str) -> float:
    aggregated_cells = pd.concat([slide.cells for slide in slides], ignore_index=True)
    number_of_cells_of_type = aggregated_cells[aggregated_cells['classification'] == cell_type].shape[0]
    total_cells = aggregated_cells.shape[0]
    return number_of_cells_of_type / (total_cells + EPSILON)

def calculate_avg_density(slides : list[PreAnnotatedSlide], cell_type : str) -> float:
    total_tissue_area = np.sum([calculate_tissue_area(slide.slide) for slide in slides])
    aggregated_cells = pd.concat([slide.cells for slide in slides], ignore_index=True)
    number_of_cells_of_type = aggregated_cells[aggregated_cells['classification'] == cell_type].shape[0]
    return number_of_cells_of_type / (total_tissue_area + EPSILON)

def calculate_avg_nn_distance(slides : list[PreAnnotatedSlide], cell_type : str) -> float:
    nn_distances = []
    for slide in slides:
        specified_cell_coords = slide.cells[slide.cells['classification'] == cell_type][['centroid_x', 'centroid_y']].to_numpy()
        if len(specified_cell_coords) == 0 or len(specified_cell_coords) == 1: continue
        tree = KDTree(specified_cell_coords)
        # the 'nearest neighbor' will always be the point itself, need the second nearest for intended behavior
        dist_in_pixels, _ = tree.query(specified_cell_coords, k=2)
        dist_in_microns = dist_in_pixels * float(slide.slide.properties['aperio.MPP'])
        nn_distances.extend(dist_in_microns[:, 1])

    return np.mean(nn_distances) if nn_distances else np.nan

In [21]:
import multiprocessing as mp

def calculate_features(study_id, slides):
    feature_dict = {}
    feature_dict['study_id'] = study_id
    feature_dict['avg_inflammatory_cell_cluster_size (um^2)'] = calculate_avg_inflammatory_cell_cluster_size(slides)
    
    for cell_type in ['epithelial-cell', 'lymphocyte', 'neutrophil', 'eosinophil', 'connective-tissue-cell', 'plasma-cell']:
        relative_abundance = calculate_relative_abundance(slides, cell_type)
        avg_density = calculate_avg_density(slides, cell_type)
        avg_nn_distance = calculate_avg_nn_distance(slides, cell_type)

        feature_dict[f'{cell_type}_relative_abundance'] = relative_abundance
        feature_dict[f'{cell_type}_avg_density (1/um^2)'] = avg_density

    return feature_dict

In [22]:
from multiprocess import Pool, cpu_count
from joblib import Parallel, delayed

if __name__ == "__main__":
    # wrap in a list to eagerly evaluate
    print("Loading slide annotations...")
    annotated_slides = {}
    for slide_path in tqdm(list(SLIDES_DIR.iterdir())):
        study_id = extract_study_id(slide_path.stem)
        corresponding_model_predictions_dir = HOVERNEXT_PREDICTIONS_DIR / slide_path.stem
        slide = PreAnnotatedSlide(slide_path, corresponding_model_predictions_dir)
        annotated_slides[study_id] = annotated_slides.get(study_id, list()) + [slide]

    nproc = cpu_count()
    features_dict = {
        'study_id' : [],
        'avg_inflammatory_cell_cluster_size (um^2)' : [],
        # relative abundances
        'epithelial-cell_relative_abundance' : [],
        'lymphocyte_relative_abundance' : [],
        'plasma-cell_relative_abundance' : [],
        'neutrophil_relative_abundance' : [],
        'eosinophil_relative_abundance' : [],
        'connective-tissue-cell_relative_abundance' : [],
        # average densities
        'epithelial-cell_avg_density (1/um^2)' : [],
        'lymphocyte_avg_density (1/um^2)' : [],
        'plasma-cell_avg_density (1/um^2)' : [],
        'neutrophil_avg_density (1/um^2)' : [],
        'eosinophil_avg_density (1/um^2)' : [],
        'connective-tissue-cell_avg_density (1/um^2)' : [],
    }
    
    with Pool(nproc) as pool:
        results = tqdm(pool.starmap(calculate_features, annotated_slides.items()), total=len(annotated_slides))
        features_dict = {k : [d[k] for d in results] for k in features_dict.keys()}
    
    features_df = pl.DataFrame(features_dict)
    features_df.write_csv(DATA_DIR / "digitized_slide_features.csv")

Loading slide annotations...


100%|██████████| 546/546 [00:11<00:00, 47.96it/s]


[14.081625985644, 14.081625985644, 7.454978462988, 7.454978462988, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581, 7.869143933154, 7.869143933154, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581, 14.081625985644, 14.081625985644, 7.454978462988, 7.454978462988, 7.869143933154, 7.869143933154, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581, 14.49579145581]
[32.581016986392, 32.581016986392, 32.581016986392, 32.581016986392, 9.249695500374, 9.249695500374, 9.249695500374, 9.249695500374, 32.581016986392, 32.581016986392, 32.581016986392, 32.581016986392, 9.249695500374, 9.249695500374, 9.249695500374, 9.249695500374, 32.581016986392, 32.581016986392, 32.581016986392, 32.581016986392, 9.249695500374, 9.249695500374, 9.249695500374, 9.249695500374, 32.581016986392, 32.581016986392, 32.581016986392, 32.581016986392, 9.249695500374, 9.249695500374, 9.249695500374, 9.249695500374]
[8.697474873486, 8

KeyboardInterrupt: 